# Labellisation des Clusters avec BirdNET

Ce notebook permet de:
1. Charger les clusters et leurs embeddings 2D
2. Selectionner 10 segments proches du centre de chaque cluster
3. Analyser ces segments avec BirdNET
4. Agreger les predictions pour labelliser chaque cluster
5. Sauvegarder les resultats dans perch2/

In [2]:
import numpy as np
import pickle
import os
import json
import io
import tempfile
from collections import defaultdict, Counter
from tqdm import tqdm
import pandas as pd

# BirdNET
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer

# Audio
import librosa
import soundfile as sf

# S3
import boto3

## 1. Configuration

In [3]:
# Chemins
PERCH2_DIR = '../perch2'
CREDENTIALS_PATH = '../credentials.json'

# Parametres
N_SAMPLES_PER_CLUSTER = 10  # Nombre de segments par cluster
MIN_CONFIDENCE = 0.5        # Seuil de confiance BirdNET
MIN_SPECIES_RATIO = 0.3     # Ratio minimum pour considerer une espece comme presente

# Categories a ignorer (bruit, pluie, etc.)
IGNORE_CATEGORIES = [
     'Noise', 
]

## 2. Chargement des donnees

In [4]:
# Charger les embeddings 2D et les labels
embeddings_2d = np.load(os.path.join(PERCH2_DIR, 'embeddings_2d_ALL.npy'))
labels = np.load(os.path.join(PERCH2_DIR, 'labels.npy'))

# Charger les metadonnees
with open(os.path.join(PERCH2_DIR, 'perch_info_FINAL.pkl'), 'rb') as f:
    metadata = pickle.load(f)

print(f"Embeddings 2D: {embeddings_2d.shape}")
print(f"Labels: {labels.shape}")
print(f"Metadata: {len(metadata)} segments")

# Statistiques des clusters
unique_labels = np.unique(labels)
n_clusters = len(unique_labels[unique_labels != -1])
n_noise = np.sum(labels == -1)
print(f"\nClusters: {n_clusters}")
print(f"Bruit (label=-1): {n_noise} ({100*n_noise/len(labels):.1f}%)")

Embeddings 2D: (650760, 2)
Labels: (650760,)
Metadata: 650760 segments

Clusters: 487
Bruit (label=-1): 99247 (15.3%)


## 3. Configuration S3

In [5]:
# Charger les credentials
with open(CREDENTIALS_PATH, 'r') as f:
    creds = json.load(f)

s3_client = boto3.client(
    's3',
    endpoint_url=creds['endpoint'],
    aws_access_key_id=creds['ACCESS_KEY'],
    aws_secret_access_key=creds['SECRET_KEY'],
    region_name=creds.get('region', 'eu-west-2')
)

BUCKET = creds['bucket']
print(f"S3 configure: {creds['endpoint']}")

S3 configure: https://s3.eu-west-2.wasabisys.com


## 4. Fonctions utilitaires

In [ ]:
def get_cluster_center_samples(cluster_id, n_samples=10):
    """
    Selectionne n_samples segments proches du centre du cluster.
    """
    # Indices des points du cluster
    cluster_mask = labels == cluster_id
    cluster_indices = np.where(cluster_mask)[0]
    
    if len(cluster_indices) == 0:
        return []
    
    # Coordonnees 2D du cluster
    cluster_coords = embeddings_2d[cluster_indices]
    
    # Centre du cluster (moyenne)
    center = cluster_coords.mean(axis=0)
    
    # Distance au centre
    distances = np.linalg.norm(cluster_coords - center, axis=1)
    
    # Trier par distance et prendre les n plus proches
    sorted_idx = np.argsort(distances)
    selected_local_idx = sorted_idx[:min(n_samples, len(sorted_idx))]
    
    # Convertir en indices globaux
    selected_global_idx = cluster_indices[selected_local_idx]
    
    return selected_global_idx.tolist()


def download_audio_segment(file_name, start_time, end_time):
    """
    Telecharge un segment audio depuis S3.
    Retourne le chemin vers un fichier temporaire.
    """
    # Construire le chemin S3
    parts = file_name.split('_')
    site = parts[0] if len(parts) >= 2 else ''
    s3_key = f"2019/{site}/Data/{file_name}"
    
    try:
        # Telecharger le fichier
        buf = io.BytesIO()
        s3_client.download_fileobj(BUCKET, s3_key, buf)
        buf.seek(0)
        
        # Charger l'audio avec librosa
        audio, sr = librosa.load(buf, sr=None, offset=start_time, duration=end_time - start_time)
        
        # Sauvegarder dans un fichier temporaire (BirdNET a besoin d'un fichier)
        temp_file = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        sf.write(temp_file.name, audio, sr)
        
        return temp_file.name, sr
    except Exception as e:
        print(f"Erreur telechargement {file_name}: {e}")
        return None, None


def analyze_with_birdnet(audio_path, analyzer):
    """
    Analyse un fichier audio avec BirdNET.
    Retourne la liste des detections.
    """
    try:
        recording = Recording(
            analyzer,
            audio_path,
            min_conf=0.1  # Seuil bas pour capturer plus de detections
        )
        recording.analyze()
        return recording.detections
    except Exception as e:
        print(f"Erreur BirdNET: {e}")
        return []


def aggregate_detections(all_detections, min_confidence=0.5, min_ratio=0.3):
    """
    Agregation des detections BirdNET pour un cluster.
    
    Retourne un dict avec:
    - species: liste des especes detectees avec leur frequence
    - main_species: espece principale (si ratio > min_ratio)
    - is_noise: True si principalement du bruit
    - confidence: confiance moyenne
    """
    if not all_detections:
        return {
            'species': [],
            'main_species': None,
            'is_noise': True,
            'confidence': 0.0,
            'n_segments_analyzed': 0
        }
    
    # Compter les especes (filtrer par confiance)
    species_counts = Counter()
    species_confidences = defaultdict(list)
    n_segments = len(all_detections)
    
    for segment_detections in all_detections:
        segment_species = set()
        for det in segment_detections:
            if det['confidence'] >= min_confidence:
                species = det['common_name']
                segment_species.add(species)
                species_confidences[species].append(det['confidence'])
        
        for sp in segment_species:
            species_counts[sp] += 1
    
    # Calculer les ratios
    species_list = []
    for species, count in species_counts.most_common():
        ratio = count / n_segments
        avg_conf = np.mean(species_confidences[species])
        species_list.append({
            'name': species,
            'count': count,
            'ratio': ratio,
            'avg_confidence': avg_conf
        })
    
    # Determiner l'espece principale
    main_species = None
    is_noise = True
    avg_confidence = 0.0
    
    # Filtrer les categories a ignorer
    valid_species = [
        sp for sp in species_list 
        if not any(ignore.lower() in sp['name'].lower() for ignore in IGNORE_CATEGORIES)
    ]
    
    if valid_species:
        top_species = valid_species[0]
        if top_species['ratio'] >= min_ratio:
            main_species = top_species['name']
            is_noise = False
            avg_confidence = top_species['avg_confidence']
    
    return {
        'species': species_list,
        'main_species': main_species,
        'is_noise': is_noise,
        'confidence': avg_confidence,
        'n_segments_analyzed': n_segments
    }

## 5. Initialisation BirdNET

In [7]:
# Initialiser l'analyseur BirdNET
print("Chargement du modele BirdNET...")
analyzer = Analyzer()
print("BirdNET pret.")

Chargement du modele BirdNET...
Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.
BirdNET pret.


c:\Users\cheml\anaconda3\envs\remi_tropical\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## 6. Analyse des clusters

In [8]:
# Liste des clusters (exclure le bruit -1)
cluster_ids = sorted([c for c in np.unique(labels) if c != -1])
print(f"Analyse de {len(cluster_ids)} clusters...")

# Resultats
cluster_labels = {}
errors = []

for cluster_id in tqdm(cluster_ids, desc="Clusters"):
    # Selectionner les segments proches du centre
    sample_indices = get_cluster_center_samples(cluster_id, N_SAMPLES_PER_CLUSTER)
    
    if not sample_indices:
        errors.append(cluster_id)
        continue
    
    # Analyser chaque segment
    all_detections = []
    
    for idx in sample_indices:
        meta = metadata[idx]
        file_name = meta['file']
        start_time = meta['start']
        end_time = meta['end']
        
        # Telecharger l'audio
        audio_path, sr = download_audio_segment(file_name, start_time, end_time)
        
        if audio_path:
            # Analyser avec BirdNET
            detections = analyze_with_birdnet(audio_path, analyzer)
            all_detections.append(detections)
            
            # Supprimer le fichier temporaire
            try:
                os.unlink(audio_path)
            except:
                pass
    
    # Agreger les detections
    result = aggregate_detections(
        all_detections, 
        min_confidence=MIN_CONFIDENCE,
        min_ratio=MIN_SPECIES_RATIO
    )
    
    # Ajouter les infos du cluster
    cluster_size = np.sum(labels == cluster_id)
    result['cluster_id'] = cluster_id
    result['cluster_size'] = cluster_size
    
    cluster_labels[cluster_id] = result

print(f"\nTermine. Erreurs: {len(errors)}")

Analyse de 487 clusters...


Clusters:   0%|          | 0/487 [00:00<?, ?it/s]

read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpjd6ks5a0.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpfn_44hft.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpurn5fy90.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpbd8m75oz.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmppfrkiamf.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp24_6tkxa.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpvfehmhtu.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp5yt6244o.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpab183n3y.wav


Clusters:   0%|          | 1/487 [00:33<4:32:56, 33.70s/it]

read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpckj1d3cg.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpama_kns_.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp82vrk4pz.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp5kznctnt.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpuszc64qo.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpijmj9s4m.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp1x4xek6n.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp_i_648ij.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpwqfo5vno.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp67oe9on5.wav


...

read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp49ar6jb0.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpf7gllbfz.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpck9n0k9g.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpgogaerfz.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpoet4o0au.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpyos4hsfw.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp253dw91q.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpsfpebhh0.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpjnsz139i.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpogy3bgou.wav


Clusters: 100%|█████████▉| 486/487 [2:27:24<00:13, 13.84s/it]

read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmptyv7of1f.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpjed6npm4.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp0405n52v.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp69wkjqxx.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmp1q5cx7u2.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpts9kmpi8.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpo9ib5tm1.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpt6zsmj62.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpcfp9zd4n.wav
read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpbuzyjphx.wav


Clusters: 100%|██████████| 487/487 [2:27:38<00:00, 18.19s/it]

read_audio_data
read_audio_data: complete, read  2 chunks.
analyze_recording tmpvjcrl5dj.wav

Termine. Erreurs: 0


## 7. Statistiques des resultats

In [9]:
# Compter les clusters avec/sans espece
n_labeled = sum(1 for c in cluster_labels.values() if c['main_species'])
n_noise = sum(1 for c in cluster_labels.values() if c['is_noise'])

print(f"Clusters avec espece identifiee: {n_labeled} ({100*n_labeled/len(cluster_labels):.1f}%)")
print(f"Clusters bruit/non-identifie: {n_noise} ({100*n_noise/len(cluster_labels):.1f}%)")

# Especes les plus frequentes
species_counter = Counter()
for c in cluster_labels.values():
    if c['main_species']:
        species_counter[c['main_species']] += 1

print("\nEspeces les plus frequentes:")
for species, count in species_counter.most_common(20):
    print(f"  {species}: {count} clusters")

Clusters avec espece identifiee: 44 (9.0%)
Clusters bruit/non-identifie: 443 (91.0%)

Especes les plus frequentes:
  Nocturnal Curassow: 12 clusters
  White-necked Thrush: 7 clusters
  Crested Owl: 3 clusters
  Snethlage's Antpitta: 3 clusters
  Capuchinbird: 2 clusters
  Fulvous Shrike-Tanager: 2 clusters
  Marail Guan: 1 clusters
  Olive-backed Foliage-gleaner: 1 clusters
  Red-necked Woodpecker: 1 clusters
  Dusky-throated Antshrike: 1 clusters
  Lined Forest-Falcon: 1 clusters
  Zigzag Heron: 1 clusters
  Spot-backed Antbird: 1 clusters
  Great Potoo: 1 clusters
  Common Scale-backed Antbird: 1 clusters
  Ferruginous-backed Antbird: 1 clusters
  Mouse-colored Antshrike: 1 clusters
  Screaming Piha: 1 clusters
  Paradise Tanager: 1 clusters
  White-throated Toucan: 1 clusters


In [10]:
# Afficher les details de quelques clusters
print("Exemples de clusters labellises:\n")

for cluster_id, info in list(cluster_labels.items())[:10]:
    print(f"Cluster {cluster_id} ({info['cluster_size']} segments):")
    if info['main_species']:
        print(f"  -> {info['main_species']} (conf: {info['confidence']:.2f})")
    else:
        print(f"  -> Non identifie / Bruit")
    
    if info['species'][:3]:
        print(f"  Detections: {[s['name'] for s in info['species'][:3]]}")
    print()

Exemples de clusters labellises:

Cluster 0 (90 segments):
  -> Marail Guan (conf: 0.71)
  Detections: ['Marail Guan']

Cluster 1 (306 segments):
  -> Non identifie / Bruit

Cluster 2 (271 segments):
  -> Non identifie / Bruit

Cluster 3 (403 segments):
  -> Non identifie / Bruit

Cluster 4 (276 segments):
  -> Non identifie / Bruit

Cluster 5 (129 segments):
  -> Non identifie / Bruit

Cluster 6 (65 segments):
  -> Non identifie / Bruit

Cluster 7 (100 segments):
  -> Non identifie / Bruit
  Detections: ['Bare-throated Tiger-Heron']

Cluster 8 (394 segments):
  -> Non identifie / Bruit

Cluster 9 (100 segments):
  -> Non identifie / Bruit



## 8. Sauvegarde des resultats

In [11]:
# Sauvegarder les labels des clusters
output_path = os.path.join(PERCH2_DIR, 'cluster_labels_birdnet.pkl')

with open(output_path, 'wb') as f:
    pickle.dump(cluster_labels, f)

print(f"Labels sauvegardes: {output_path}")

# Sauvegarder aussi en CSV pour lecture facile
csv_data = []
for cluster_id, info in cluster_labels.items():
    csv_data.append({
        'cluster_id': cluster_id,
        'cluster_size': info['cluster_size'],
        'main_species': info['main_species'] or 'Unknown',
        'confidence': info['confidence'],
        'is_noise': info['is_noise'],
        'n_analyzed': info['n_segments_analyzed'],
        'all_species': '|'.join([s['name'] for s in info['species'][:5]])
    })

df = pd.DataFrame(csv_data)
csv_path = os.path.join(PERCH2_DIR, 'cluster_labels_birdnet.csv')
df.to_csv(csv_path, index=False)
print(f"CSV sauvegarde: {csv_path}")

Labels sauvegardes: ../perch2\cluster_labels_birdnet.pkl
CSV sauvegarde: ../perch2\cluster_labels_birdnet.csv


In [13]:
# Creer un mapping cluster_id -> label simple pour l'UI
cluster_to_label = {}

for cluster_id, info in cluster_labels.items():
    if info['main_species']:
        cluster_to_label[cluster_id] = info['main_species']
    else:
        cluster_to_label[cluster_id] = 'Unknown'

# Sauvegarder le mapping simple
mapping_path = os.path.join(PERCH2_DIR, 'cluster_species_mapping.pkl')
with open(mapping_path, 'wb') as f:
    pickle.dump(cluster_to_label, f)

print(f"Mapping sauvegarde: {mapping_path}")
print(f"\nExemple du mapping:")
for k, v in list(cluster_to_label.items())[:10]:
    print(f"  Cluster {k} -> {v}")

Mapping sauvegarde: ../perch2\cluster_species_mapping.pkl

Exemple du mapping:
  Cluster 0 -> Marail Guan
  Cluster 1 -> Unknown
  Cluster 2 -> Unknown
  Cluster 3 -> Unknown
  Cluster 4 -> Unknown
  Cluster 5 -> Unknown
  Cluster 6 -> Unknown
  Cluster 7 -> Unknown
  Cluster 8 -> Unknown
  Cluster 9 -> Unknown


## 9. Resume

In [14]:
print("=" * 50)
print("RESUME DE L'ANALYSE")
print("=" * 50)
print(f"\nClusters analyses: {len(cluster_labels)}")
print(f"Clusters avec espece: {n_labeled}")
print(f"Clusters bruit/inconnu: {n_noise}")
print(f"\nEspeces uniques detectees: {len(species_counter)}")
print(f"\nFichiers sauvegardes:")
print(f"  - {output_path}")
print(f"  - {csv_path}")
print(f"  - {mapping_path}")

RESUME DE L'ANALYSE

Clusters analyses: 487
Clusters avec espece: 44
Clusters bruit/inconnu: 443

Especes uniques detectees: 21

Fichiers sauvegardes:
  - ../perch2\cluster_labels_birdnet.pkl
  - ../perch2\cluster_labels_birdnet.csv
  - ../perch2\cluster_species_mapping.pkl
